# SuSchedule-r — Build FAISS index over SU course catalog

Encodes every course in `SU_full_catalog.json` with **BAAI/bge-base-en-v1.5** and builds a FAISS `IndexFlatIP` (exact cosine search, since the corpus is only 688 docs).

**Outputs (saved to `/content/out/` and downloaded at the end):**
- `su_courses.parquet` — the structured DataFrame (one row per course)
- `su_courses_embeddings.npy` — float32 embedding matrix `(N, 768)`, L2-normalized
- `su_courses.index` — FAISS `IndexFlatIP` ready to `faiss.read_index(...)`
- `id_map.json` — row index → course code (FAISS returns positional ids)

Runtime: ~1–2 minutes on a free T4 GPU runtime.

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu pandas pyarrow

## 2. Load the catalog

Pulls `SU_full_catalog.json` straight from the `kero` branch on GitHub. If the URL ever breaks (private fork, branch renamed), replace the cell with a manual `files.upload()`.

In [ ]:
import json, urllib.request, pathlib

CATALOG_URL = "https://raw.githubusercontent.com/keremsirtikizil/SuSchedule-r/kero/data/SU_full_catalog.json"
LOCAL_PATH = pathlib.Path("/content/SU_full_catalog.json")

if not LOCAL_PATH.exists():
    print(f"Fetching {CATALOG_URL} ...")
    urllib.request.urlretrieve(CATALOG_URL, LOCAL_PATH)

catalog = json.loads(LOCAL_PATH.read_text())
print(f"Loaded {catalog['course_count']} courses across {len(catalog['degrees'])} degrees.")

## 3. Build the DataFrame

One row per course. Splits cleanly into three groups:
- **identity**: `code`, `subj`, `num`, `level`, `title`
- **encoder input**: `description`, `embedding_text`
- **filter metadata**: `su_credit`, `ects`, `has_prereq`, `prereq_text`, `programs`, `sections`, `seasons_offered`, `last_offered_term`, `is_active`

In [ ]:
import re
import pandas as pd

EMPTY_PREREQ_MARKERS = {"", "__"}

_ects_re = re.compile(r"(\d+(?:\.\d+)?)\s*ECTS", re.IGNORECASE)
_term_season_re = re.compile(r"(Fall|Spring|Summer)\s+(\d{4})-(\d{4})", re.IGNORECASE)
_SEASON_CODE = {"fall": "01", "spring": "02", "summer": "03"}


def parse_ects(text):
    if not text:
        return None
    m = _ects_re.search(text)
    return float(m.group(1)) if m else None


def term_to_code(term_name):
    """'Fall 2025-2026' -> '202501' so we can compare lexicographically."""
    m = _term_season_re.search(term_name or "")
    if not m:
        return None
    season, start_year, _ = m.groups()
    return f"{start_year}{_SEASON_CODE[season.lower()]}"


def derive_offering_info(offered_terms):
    seasons, codes = set(), []
    for entry in offered_terms or []:
        name = entry.get("term", "")
        m = _term_season_re.search(name)
        if m:
            seasons.add(m.group(1).capitalize())
        code = term_to_code(name)
        if code:
            codes.append(code)
    seasons_sorted = sorted(seasons, key=["Fall", "Spring", "Summer"].index)
    last_term = max(codes) if codes else None
    return seasons_sorted, last_term


# threshold for is_active: ran in any of the last 4 terms (= ~1.5 academic years)
ACTIVE_CUTOFF = "202403"  # anything >= summer 2024-2025

rows = []
for c in catalog["courses"]:
    num_raw = c.get("num") or "0"
    try:
        num = int(re.sub(r"\D", "", num_raw) or 0)
    except ValueError:
        num = 0
    level = num // 100 if num else 0
    title = (c.get("title") or "").strip()
    description = (c.get("description") or "").strip()
    prereq_text = (c.get("prereq_text") or "").strip()
    has_prereq = prereq_text not in EMPTY_PREREQ_MARKERS

    programs, sections = set(), set()
    for ps in c.get("program_sections") or []:
        if ps.get("program"):
            programs.add(ps["program"])
        if ps.get("section"):
            sections.add(ps["section"])

    seasons_offered, last_offered_term = derive_offering_info(c.get("offered_terms"))
    is_active = last_offered_term is not None and last_offered_term >= ACTIVE_CUTOFF

    # embedding_text — only code/title/description, per project preference
    if description:
        embedding_text = f"{c['code']} \u2014 {title}\n{description}"
    else:
        embedding_text = f"{c['code']} \u2014 {title}"

    rows.append({
        "code": c["code"],
        "subj": c.get("subj", ""),
        "num": num,
        "level": level,
        "title": title,
        "description": description,
        "embedding_text": embedding_text,
        "su_credit": c.get("su_credit"),
        "ects": parse_ects(c.get("ects_text")),
        "has_prereq": has_prereq,
        "prereq_text": prereq_text if has_prereq else "",
        "programs": sorted(programs),
        "sections": sorted(sections),
        "seasons_offered": seasons_offered,
        "last_offered_term": last_offered_term,
        "is_active": is_active,
    })

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
df.head(3)

In [ ]:
# quick sanity
print("Courses missing description :", (df["description"] == "").sum())
print("Courses missing title       :", (df["title"] == "").sum())
print("Active in last 4 terms      :", df["is_active"].sum(), "/", len(df))
print("With prereq                 :", df["has_prereq"].sum())
print("Embedding text length stats :")
print(df["embedding_text"].str.len().describe().round(1))

## 4. Encode with `BAAI/bge-base-en-v1.5`

BGE recommends a short instruction prefix for **queries**; passages (our courses) are encoded plain. Output is L2-normalized so inner-product == cosine.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "BAAI/bge-base-en-v1.5"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading model on", device)

model = SentenceTransformer(MODEL_NAME, device=device)
print("Embedding dim:", model.get_sentence_embedding_dimension())

In [ ]:
passages = df["embedding_text"].tolist()

embeddings = model.encode(
    passages,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,   # -> inner product == cosine
    convert_to_numpy=True,
).astype("float32")

print("Embeddings shape:", embeddings.shape)
# row norms should all be ~1.0
print("Sample norms:", np.linalg.norm(embeddings[:5], axis=1).round(4))

## 5. Build the FAISS index

`IndexFlatIP` does an exact dot product against every vector. At 688 × 768 that's a few ms per query — no need for IVF/HNSW.

In [ ]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print("Index size:", index.ntotal)

## 6. Sanity-check with a sample query

Encode the query with the BGE retrieval prefix, then search the top-k.

In [ ]:
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def search(query: str, k: int = 5) -> pd.DataFrame:
    q_vec = model.encode(
        [BGE_QUERY_PREFIX + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    scores, idxs = index.search(q_vec, k)
    hits = df.iloc[idxs[0]].copy()
    hits.insert(0, "score", scores[0])
    return hits[["score", "code", "title", "subj", "level"]]

search("introduction to machine learning and neural networks", k=5)

In [ ]:
search("financial accounting for managers", k=5)

In [ ]:
search("differential equations and linear algebra", k=5)

## 7. Save everything

In [ ]:
OUT_DIR = pathlib.Path("/content/out")
OUT_DIR.mkdir(exist_ok=True)

df.to_parquet(OUT_DIR / "su_courses.parquet", index=False)
np.save(OUT_DIR / "su_courses_embeddings.npy", embeddings)
faiss.write_index(index, str(OUT_DIR / "su_courses.index"))

id_map = {i: code for i, code in enumerate(df["code"].tolist())}
(OUT_DIR / "id_map.json").write_text(json.dumps(id_map, indent=2))

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:35s} {p.stat().st_size/1024:8.1f} KB")

In [ ]:
# Download the artifacts to your local machine.
from google.colab import files
for p in sorted(OUT_DIR.iterdir()):
    files.download(str(p))

## 8. How to reload locally

```python
import faiss, json, pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer

df    = pd.read_parquet("su_courses.parquet")
index = faiss.read_index("su_courses.index")
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

q = "Represent this sentence for searching relevant passages: distributed systems"
v = model.encode([q], normalize_embeddings=True).astype("float32")
scores, idxs = index.search(v, 5)
print(df.iloc[idxs[0]][["code", "title"]])
```